# Notebook 02 · L0：静态结构依赖与一阶/二阶相变

**L0 = 布尔耦合 + GCC 剪枝**：最朴素的相依网络模型，也是所有台阶的基线。

这一节复现的是 **PNAS 2019 图 3A** 的那句话：
> 相依网络的崩溃是**一阶相变**（突然跳落），而单层网络是二阶（连续下降）。

两个旋钮：耦合强度 `q`（coupled_fraction）与移除比例 `f`。

## 环境准备（每个 notebook 都要先跑这两格）

In [ ]:
"""环境准备：定位项目根目录、加 src 到搜索路径、设好画图参数。"""
import sys
from pathlib import Path

# 从当前工作目录往上找，直到看见 src/phase6_interdependent（这样在 notebooks/ 里也能跑）
root = Path.cwd()
while not (root / "src" / "phase6_interdependent").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

RESULTS = root / "results"
RESULTS.mkdir(exist_ok=True)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("项目根目录：", root)

In [ ]:
# 公共底座：一对 BA 双层网络 + 骨架类（后面所有 notebook 都从这一格开始）
from phase6_interdependent.core import InterdependentNetwork

N, m = 300, 3
G_A = nx.barabasi_albert_graph(N, m, seed=42)   # 层 A：比如电力网
G_B = nx.barabasi_albert_graph(N, m, seed=43)   # 层 B：比如通信网
nodes_A = sorted(G_A.nodes())
print(f"双层网络就绪：N={N}, 每层边数={G_A.number_of_edges()}")

## 第 1 步

In [ ]:
from phase6_interdependent.L0_static_dependency import (
    percolation_curve, single_layer_curve, transition_order, run_experiment
)

fractions = np.linspace(0.0, 0.5, 26)
_, S_single = single_layer_curve(G_A, fractions, n_trials=3, seed=42)
_, S_weak = percolation_curve(G_A, G_B, fractions, coupled_fraction=0.2, n_trials=3, seed=42)
_, S_strong = percolation_curve(G_A, G_B, fractions, coupled_fraction=1.0, n_trials=3, seed=42)

for name, S in [("单层", S_single), ("双层 q=0.2", S_weak), ("双层 q=1.0", S_strong)]:
    order, jump = transition_order(fractions, S)
    print(f"{name:>10}：相变={order}，最大单步跌幅={jump:.3f}")

## 第 2 步

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(fractions, S_single, "o-", label="单层网络（对照）")
plt.plot(fractions, S_weak, "s-", label="双层 · 弱耦合 q=0.2")
plt.plot(fractions, S_strong, "^-", label="双层 · 强耦合 q=1.0")
plt.xlabel("移除节点比例 f")
plt.ylabel("存活序参量 S")
plt.title("L0：层间耦合把二阶相变变成一阶")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(RESULTS / "notebook_02_L0_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 第 3 步

In [ ]:
# 一键跑完整实验（会把图存到 results/）
out = run_experiment(N=300, m=3, n_trials=5, seed=42, save=True)
print("弱耦合相变：", out["order_weak"], " 强耦合相变：", out["order_strong"])

**思考题**

1. 为什么单层网络是二阶相变，而双层相依网络是一阶？（提示：正反馈）
2. `q` 从 1.0 降到 0.2 时曲线怎么变？这对应 PNAS 2019 的哪个结论？
3. 如果两层不是一一对应，而是"多个 A 依赖一个 B"，p_c 会怎么变？